# Ingestion of data into the Bronze layer
This Notebook will focus on ingesting the .RData files as is as bronze tables
## Introduction
The data is stored in 4 .Rdata files that need to be read and then stored as tables
## Code
This notebook will be written in python
### Libraries
Pandas and pyreadr will be installed to operate with dataframes and .RData files.

In [0]:
%pip install pandas
%pip install pyreadr

## Paths
Here you can read the paths of the different files and tables together with the source of the data

In [0]:
catalog = "tep_lakehouse"
schema = "bronze"
volume = "raw_ingest"
download_url = "https://www.kaggle.com/datasets/averkij/tennessee-eastman-process-simulation-dataset"
table_name = ["bronze_te_ff","bronze_tr_ff","bronze_te_f","bronze_tr_f"]
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema
print(path_table) # Show the complete path
print(path_volume) # Show the complete path

### Import files
Data will be imported as **objects**

In [0]:
import pyreadr

bronze_te_ff_raw = pyreadr.read_r('/Volumes/tep_lakehouse/bronze/raw_ingest/TEP_FaultFree_Testing.RData')
bronze_te_f_raw = pyreadr.read_r('/Volumes/tep_lakehouse/bronze/raw_ingest/TEP_Faulty_Testing.RData')
bronze_tr_ff_raw = pyreadr.read_r('/Volumes/tep_lakehouse/bronze/raw_ingest/TEP_FaultFree_Training.RData')
bronze_tr_f_raw = pyreadr.read_r('/Volumes/tep_lakehouse/bronze/raw_ingest/TEP_Faulty_Training.RData')

### Transformations
- In order for the data to be easily ingested, it needs to be turned into parquet files.
- Afterwards, the schema defined bellow will be applied to that data
- The metadata columns will be added to be able to audit the data flow
- Finally, It'll be stored as a table.

In [0]:

my_schema ="faultNumber int, simulationRun int, sample int, xmeas_1 double, xmeas_2 double, xmeas_3 double, xmeas_4 double, xmeas_5 double, xmeas_6 double, xmeas_7 double, xmeas_8 double, xmeas_9 double, xmeas_10 double, xmeas_11 double, xmeas_12 double, xmeas_13 double, xmeas_14 double, xmeas_15 double, xmeas_16 double, xmeas_17 double, xmeas_18 double, xmeas_19 double, xmeas_20 double, xmeas_21 double, xmeas_22 double, xmeas_23 double, xmeas_24 double, xmeas_25 double, xmeas_26 double, xmeas_27 double, xmeas_28 double, xmeas_29 double, xmeas_30 double, xmeas_31 double, xmeas_32 double, xmeas_33 double, xmeas_34 double, xmeas_35 double, xmeas_36 double, xmeas_37 double, xmeas_38 double, xmeas_39 double, xmeas_40 double, xmeas_41 double, xmv_1 double, xmv_2 double, xmv_3 double, xmv_4 double, xmv_5 double, xmv_6 double, xmv_7 double, xmv_8 double, xmv_9 double, xmv_10 double, xmv_11 double"
schema = "tep_lakehouse.bronze"
temp_base = "/Volumes/tep_lakehouse/bronze/raw_ingest/tmp_parquet"


_Since this is a one time ingest, I don't need to thing the type of ingestion, it doesn't need updating or overwriting_

In [0]:
import os
from pyspark.sql.types import StructType
from pyspark.sql.functions import col, current_timestamp

os.makedirs(temp_base, exist_ok=True)
spark_schema = StructType.fromDDL(my_schema)

datasets = {
    "tep_faultfree_testing":  bronze_te_ff_raw,
    "tep_faulty_testing":     bronze_te_f_raw,
    "tep_faultfree_training": bronze_tr_ff_raw,
    "tep_faulty_training":    bronze_tr_f_raw,
}

for table_name, result in datasets.items():
    for r_obj_name, df in result.items():
        print(f"Processing {r_obj_name} | shape: {df.shape}")

        for col_name in ["faultNumber", "simulationRun", "sample"]:
            if col_name in df.columns:
                df[col_name] = df[col_name].astype("Int32")

        temp_path = f"{temp_base}/{table_name}.parquet"
        df.to_parquet(temp_path, index=False)

        full_table_name = f"tep_lakehouse.bronze.{table_name}"
        print(f"  Writing to: {full_table_name}")

        spark_df = spark.read.schema(spark_schema).parquet(temp_path)

        spark_df_with_metadata = (
            spark_df.withColumn("source_file", col("_metadata.file_name"))
            .withColumn("ingestion_ts", current_timestamp())
)

        spark_df_with_metadata.writeTo(full_table_name) \
            .using("delta") \
            .tableProperty("overwriteSchema", "true") \
            .createOrReplace()
        
        

        print(f"  Done: {full_table_name}")
        os.remove(temp_path)

print("All done.")

Quality checks

In [0]:
# bronze_quality_log.py
# Utility: log row count + per-column null rates to <catalog>.bronze.quality_log
# Schema: long format — one row per (table, column, ingestion run)
#
# Usage:
#   from bronze_quality_log import create_quality_log_table, log_bronze_quality
#
#   create_quality_log_table(catalog="my_catalog")
#   log_bronze_quality(df=bronze_df, table_name="tep_process", catalog="my_catalog")

from datetime import datetime, timezone

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

QUALITY_LOG_SCHEMA = StructType(
    [
        StructField("table_name", StringType(), nullable=False),
        StructField("ingestion_timestamp", TimestampType(), nullable=False),
        StructField("row_count", LongType(), nullable=False),
        StructField("column_name", StringType(), nullable=False),
        StructField("null_count", LongType(), nullable=False),
        StructField("null_rate", DoubleType(), nullable=True),  # None when row_count = 0
    ]
)


def create_quality_log_table(
    catalog: str,
    bronze_schema: str = "bronze",
    spark: SparkSession | None = None,
) -> None:
    """
    Create <catalog>.<bronze_schema>.quality_log as a managed Delta table.
    Safe to call repeatedly — uses CREATE TABLE IF NOT EXISTS.
    """
    spark = spark or SparkSession.getActiveSession()
    fqn = f"`{catalog}`.`{bronze_schema}`.`quality_log`"

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {fqn} (
            table_name          STRING    NOT NULL COMMENT 'Source Bronze table name',
            ingestion_timestamp TIMESTAMP NOT NULL COMMENT 'UTC timestamp of the ingestion run',
            row_count           BIGINT    NOT NULL COMMENT 'Total rows in the Bronze table at ingestion',
            column_name         STRING    NOT NULL COMMENT 'Column being profiled',
            null_count          BIGINT    NOT NULL COMMENT 'Number of null values in this column',
            null_rate           DOUBLE             COMMENT 'Fraction of nulls (null_count / row_count)'
        )
        USING DELTA
        COMMENT 'Bronze layer quality log — row counts and null rates per column per ingestion run'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """)
    print(f"[quality_log] Table ready: {fqn}")


def log_bronze_quality(
    df: DataFrame,
    table_name: str,
    catalog: str,
    bronze_schema: str = "bronze",
    spark: SparkSession | None = None,
) -> None:
    """
    Profile a Bronze DataFrame and append quality metrics to the quality_log table.

    Computes in a single pass over the data:
      - Total row count
      - Null count and null rate per column
      - UTC ingestion timestamp

    Args:
        df:            The Bronze DataFrame to profile (post-ingestion, pre-transform).
        table_name:    Logical name stored in the log (e.g. "tep_process_data").
        catalog:       Unity Catalog catalog name.
        bronze_schema: Schema name (default: "bronze").
        spark:         SparkSession — auto-detected if not provided.
    """
    spark = spark or SparkSession.getActiveSession()
    fqn = f"`{catalog}`.`{bronze_schema}`.`quality_log`"
    ingestion_ts = datetime.now(timezone.utc).replace(tzinfo=None)  # Spark Timestamp is tz-naive

    # Single pass: count nulls for all columns simultaneously
    null_exprs = [
        F.sum(F.col(c).isNull().cast("long")).alias(c)
        for c in df.columns
    ]
    row_count: int = df.count()
    null_counts: dict[str, int] = df.select(null_exprs).collect()[0].asDict()

    records = [
        {
            "table_name": table_name,
            "ingestion_timestamp": ingestion_ts,
            "row_count": row_count,
            "column_name": col_name,
            "null_count": null_count,
            "null_rate": round(null_count / row_count, 6) if row_count > 0 else None,
        }
        for col_name, null_count in null_counts.items()
    ]

    quality_df = spark.createDataFrame(records, schema=QUALITY_LOG_SCHEMA)

    (
        quality_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(fqn)
    )

    # Surface any columns with high null rates for immediate visibility
    flagged = [r for r in records if r["null_rate"] is not None and r["null_rate"] > 0.05]
    flag_msg = (
        f" | ⚠ {len(flagged)} column(s) >5% nulls: {[r['column_name'] for r in flagged]}"
        if flagged else ""
    )
    print(f"[quality_log] {table_name}: {row_count:,} rows | {len(records)} columns logged{flag_msg}")


# ---------------------------------------------------------------------------
    # Step 1: create the table once
    create_quality_log_table(catalog=catalog)

In [0]:

bronze_df = spark.read.table(f"`{catalog}`.bronze.tep_faultfree_training")
log_bronze_quality(df=bronze_df, table_name="tep_faultfree_training", catalog=catalog)

bronze_df = spark.read.table(f"`{catalog}`.bronze.tep_faultfree_testing")
log_bronze_quality(df=bronze_df, table_name="tep_faultfree_testing", catalog=catalog)

bronze_df = spark.read.table(f"`{catalog}`.bronze.tep_faulty_testing")
log_bronze_quality(df=bronze_df, table_name="tep_faulty_testing", catalog=catalog)

bronze_df = spark.read.table(f"`{catalog}`.bronze.tep_faulty_training")
log_bronze_quality(df=bronze_df, table_name="tep_faulty_training", catalog=catalog)